# R07-H50 - CFAR on the drift stream: formal trial of the rejected candidate

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-06 <br>
**Pipeline stage**: R07 contrarian slate, steelman of the H33 CFAR design note <br>
**Data**: realized wave-1b post-cure remap series (neo4j3 control metanode) + Binomial synthetic baseline <br>

The H33 note rejected radar-style CFAR by argument. This notebook gives it the fair trial: every detector calibrated to the SAME false-alarm budget (unequal budgets are the classic way detector comparisons lie), then raced on injected sustained remap shifts - the drift detector's actual target.

Detectors (one free scalar each, tuned by bisection to target ARL0):
- **production** - alarm when all last-3 rates exceed a fixed threshold (the shipped criterion)
- **CA-CFAR** - alarm when the rate exceeds alpha x trailing-window mean (window 16, guard 2)
- **OS-CFAR** - alpha x trailing-window 75th percentile (rank-based, clutter-robust variant)
- **CFAR 2-of-3** - binary integration over CA-CFAR exceedances (the sustained-target variant radar practice would deploy)
- **CUSUM** - one-sided cumulative sum; reference mu0 adapts by EWMA ONLY while the statistic is at zero (quiescent adaptation), so accumulating evidence can never raise its own reference

$$S_i = \max(0,\ S_{i-1} + x_i - (\hat{\mu}_0 + \delta/2)), \quad \text{alarm at } S_i > h$$

**Iteration 2 corrections** (iteration 1's calibration was degenerate, and each failure is evidence in its own right):
- ARL0 was estimated on streams equal in length to the acceptance band's upper edge, so a detector that NEVER alarms passed as calibrated - calibration streams are now 5x the target so censoring cannot masquerade as calibration
- the epsilon floor on CFAR's reference statistic silently turned OS-CFAR into a fixed-threshold detector on quiet noise floors (alpha x eps = const) - the zero-floor pathology the H33 note predicted, now reported instead of hidden
- CUSUM's reference was a trailing median, which rises during a sustained shift and masks it - the exact disease attributed to CFAR; replaced with quiescent-only adaptation
- iteration 1's real-stream check already showed the point-detector signature: both CFAR variants alarmed on the clean 246-doc series' single spike document; production, 2-of-3 and CUSUM stayed silent

## Approach
1. **Load** - the realized 246-doc post-cure remap series (clean-stream silence check) and the realized entities-per-doc distribution driving the Binomial noise model (small-denominator burstiness reproduced by construction)
2. **Calibrate** - bisection on each detector's scalar until empirical ARL0 hits the ~500-doc target on long clean streams
3. **Race** - step (+0.30) and ramp (10-doc onset) shifts at doc 300, 200 trials each, 40-doc horizon; delay and miss rate; repeated at baseline p0 = 0.01 / 0.05 / 0.10
4. **Verdict** - CFAR refuted if every variant shows >= 2x CUSUM's ramp miss rate or >= 2x its step delay; vindicated (H33 note superseded) if the best variant lands within 25% on both

## Outputs
- `reports/drift-cfar-h50-<stamp>.json` - calibrated scalars, per-detector delay/miss grids, clean-stream alarms, verdict
- In-notebook: calibration table, race results per noise level, verdict

In [1]:
# Imports
# stdlib
import datetime  # report stamps
import json  # report persistence
import os  # graph selection env

# third party
import numpy as np  # simulation + detectors
from numpy.lib.stride_tricks import sliding_window_view  # vectorized windows
from pathlib import Path
from rich import print as rprint  # semantic output
from rich.progress import Progress  # calibration + race loops
from neo4j import GraphDatabase  # realized series + entity counts

os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j3:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

# project
from knowledge_graph_foundry.graph.metanode import read_control  # drift detector state

2026-07-06 20:37:40.631 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# Reproducibility - all simulation randomness from one seeded generator
rng = np.random.default_rng(42)

In [3]:
# Configuration
ARL0_TARGET = 500          # docs between false alarms, all detectors matched here
ARL0_TOL = (420, 600)      # acceptance band for calibration
N_CAL_STREAMS = 150        # clean streams per calibration probe
CAL_LEN = 2500             # calibration stream length: 5x target so censoring cannot pass as calibration
BISECT_ITERS = 18          # bisection depth per detector

STREAM_LEN = 400           # race stream length
N_TRIALS = 200             # shift trials per detector per scenario
ONSET = 300                # shift onset doc
HORIZON = 40               # docs after onset before a miss is declared
SHIFT_DELTA = 0.30         # step magnitude in remap probability
RAMP_LEN = 10              # docs over which the ramp reaches full delta
P0_GRID = [0.01, 0.05, 0.10]  # baseline remap probabilities (noise floors)

CFAR_WINDOW = 16           # trailing reference cells
CFAR_GUARD = 2             # guard cells between test cell and reference
CFAR_EPS = 1e-3            # noise-floor epsilon (zero-mean reference pathology, reported when binding)
OS_QUANTILE = 75           # OS-CFAR order statistic
PROD_WINDOW = 3            # production all-consecutive window
CUSUM_ALLOWANCE = SHIFT_DELTA / 2  # design-shift allowance k
CUSUM_EWMA = 0.05          # quiescent-only reference adaptation rate

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"\u2500" * 40}[/dim]
[bold]False-alarm matching[/bold]
  ARL0 target: [yellow]{ARL0_TARGET}[/yellow] docs [dim](band {ARL0_TOL}, calibration streams {CAL_LEN} docs - censoring cannot pass)[/dim]
  Calibration: [yellow]{N_CAL_STREAMS}[/yellow] streams, [yellow]{BISECT_ITERS}[/yellow] bisection steps

[bold]Shift race[/bold]
  Step: +[yellow]{SHIFT_DELTA}[/yellow]  Ramp: [yellow]{RAMP_LEN}[/yellow] docs to full delta
  Trials: [yellow]{N_TRIALS}[/yellow]  Onset: doc [yellow]{ONSET}[/yellow]  Horizon: [yellow]{HORIZON}[/yellow]
  Noise floors p0: [yellow]{P0_GRID}[/yellow]

[bold]Detector structure[/bold]
  CFAR: window [yellow]{CFAR_WINDOW}[/yellow], guard [yellow]{CFAR_GUARD}[/yellow], OS quantile [yellow]{OS_QUANTILE}[/yellow]
  Production window: [yellow]{PROD_WINDOW}[/yellow]  CUSUM k: [yellow]{CUSUM_ALLOWANCE}[/yellow], quiescent EWMA: [yellow]{CUSUM_EWMA}[/yellow]
""")

Configuration
────────────────────────────────────────
False-alarm matching
  ARL0 target: 500 docs (band (420, 600), calibration streams 2500 docs - censoring cannot pass)
  Calibration: 150 streams, 18 bisection steps

Shift race
  Step: +0.3  Ramp: 10 docs to full delta
  Trials: 200  Onset: doc 300  Horizon: 40
  Noise floors p0: [0.01, 0.05, 0.1]

Detector structure
  CFAR: window 16, guard 2, OS quantile 75
  Production window: 3  CUSUM k: 0.15, quiescent EWMA: 0.05

## Data loading

The realized post-cure remap series (drift detector state persisted in the wave-1b control metanode) is the clean-stream silence check - the campaign monitoring confirms no genuine content shift, so any alarm on it is a false alarm. The realized entities-per-doc distribution drives the Binomial noise model: remaps_i ~ Bin(n_i, p), x_i = remaps_i / n_i - which reproduces the small-denominator burstiness (a 2-entity doc jumps straight to rate 0.5) that a Gaussian noise model would hide.

In [4]:
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
    notifications_min_severity="OFF",
)
state = read_control(driver) or {}
real_series = np.array((state.get("drift") or {}).get("remap_rates", []))
with driver.session() as s:
    doc_counts = np.array([
        r["n"] for r in s.run(
            "MATCH (d:KGFDocument) "
            "OPTIONAL MATCH (e:Entity) WHERE d.id IN e.source_documents "
            "RETURN d.id AS doc, count(e) AS n"
        ) if r["n"] > 0
    ])
driver.close()

rprint(
    f"realized remap series: [yellow]{len(real_series)}[/yellow] docs, "
    f"nonzero [yellow]{int((real_series > 0).sum())}[/yellow], mean [yellow]{real_series.mean():.4f}[/yellow]\n"
    f"entities-per-doc: [yellow]{len(doc_counts)}[/yellow] docs, "
    f"median [yellow]{int(np.median(doc_counts))}[/yellow], min [yellow]{doc_counts.min()}[/yellow], "
    f"p10 [yellow]{int(np.percentile(doc_counts, 10))}[/yellow] [dim](small denominators are real)[/dim]"
)


def simulate_stream(p_path, rng):
    """Binomial remap stream over the realized entities-per-doc distribution."""
    n = rng.choice(doc_counts, size=len(p_path))
    return rng.binomial(n, p_path) / n

realized remap series: 255 docs, nonzero 1, mean 0.0039
entities-per-doc: 262 docs, median 13, min 3, p10 9 (small denominators are real)

## Detectors and false-alarm calibration

Each detector maps a stream to a sorted array of alarm indices (CUSUM resets its statistic on alarm - the production semantic: alarm, investigate, resume). Calibration bisects the single free scalar until ARL0 (mean docs to FIRST alarm, on 2500-doc clean streams) lands in the band; a fully-silent detector reads ARL0 = 2500, far outside the band, so silence can no longer pass as calibration. CFAR reference statistics keep the epsilon floor - when it binds, CFAR degenerates into a fixed threshold, and the calibrated alpha reveals it.

In [5]:
def alarms_production(x, theta):
    w = sliding_window_view(x, PROD_WINDOW)
    mask = np.zeros(len(x), dtype=bool)
    mask[PROD_WINDOW - 1:] = (w > theta).all(axis=1)
    return np.flatnonzero(mask)


def _cfar_exceed(x, alpha, os_quantile=None):
    lead = CFAR_WINDOW + CFAR_GUARD
    mask = np.zeros(len(x), dtype=bool)
    if len(x) <= lead:
        return mask
    refs = sliding_window_view(x, CFAR_WINDOW)
    stat = (
        np.percentile(refs, os_quantile, axis=1) if os_quantile is not None
        else refs.mean(axis=1)
    )
    thr = alpha * np.maximum(stat[: len(x) - lead], CFAR_EPS)
    mask[lead:] = x[lead:] > thr
    return mask


def alarms_ca_cfar(x, alpha):
    return np.flatnonzero(_cfar_exceed(x, alpha))


def alarms_os_cfar(x, alpha):
    return np.flatnonzero(_cfar_exceed(x, alpha, os_quantile=OS_QUANTILE))


def alarms_cfar_2of3(x, alpha):
    e = _cfar_exceed(x, alpha).astype(int)
    if len(e) < 3:
        return np.array([], dtype=int)
    w = sliding_window_view(e, 3).sum(axis=1)
    mask = np.zeros(len(x), dtype=bool)
    mask[2:] = w >= 2
    return np.flatnonzero(mask)


def alarms_cusum(x, h):
    """Quiescent-adaptation CUSUM: mu0 EWMA-updates only while S == 0, so
    accumulating evidence never raises its own reference (the self-masking
    trap the trailing-median iteration-1 version fell into)."""
    alarms = []
    s, mu0 = 0.0, 0.0
    for i, xi in enumerate(x):
        if s == 0.0:
            mu0 = (1 - CUSUM_EWMA) * mu0 + CUSUM_EWMA * xi
        s = max(0.0, s + xi - (mu0 + CUSUM_ALLOWANCE))
        if s > h:
            alarms.append(i)
            s = 0.0
    return np.array(alarms, dtype=int)


DETECTORS = {
    "production": (alarms_production, 0.02, 1.0),
    "ca_cfar": (alarms_ca_cfar, 1.0, 400.0),
    "os_cfar": (alarms_os_cfar, 1.0, 400.0),
    "cfar_2of3": (alarms_cfar_2of3, 1.0, 400.0),
    "cusum": (alarms_cusum, 0.005, 4.0),
}


def empirical_arl0(alarm_fn, scalar, streams):
    firsts = []
    for x in streams:
        a = alarm_fn(x, scalar)
        firsts.append(a[0] + 1 if len(a) else len(x))
    return float(np.mean(firsts))


calibrated = {}
with Progress() as progress:
    task = progress.add_task("calibrating", total=len(P0_GRID) * len(DETECTORS))
    for p0 in P0_GRID:
        streams = [simulate_stream(np.full(CAL_LEN, p0), rng) for _ in range(N_CAL_STREAMS)]
        for name, (fn, lo, hi) in DETECTORS.items():
            scalar, arl = None, None
            for _ in range(BISECT_ITERS):
                mid = (lo + hi) / 2
                arl = empirical_arl0(fn, mid, streams)
                scalar = mid
                if ARL0_TOL[0] <= arl <= ARL0_TOL[1]:
                    break
                if arl < ARL0_TARGET:
                    lo = mid
                else:
                    hi = mid
            in_band = ARL0_TOL[0] <= arl <= ARL0_TOL[1]
            calibrated[(p0, name)] = {"scalar": scalar, "arl0": arl, "in_band": in_band}
            progress.console.print(
                f"p0={p0} {name}: scalar={scalar:.4f} ARL0={arl:.0f}"
                + ("" if in_band else "  UNCALIBRATABLE (bound hit)")
            )
            progress.advance(task)

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

p0=0.01 production: scalar=0.0200 ARL0=624  UNCALIBRATABLE (bound hit)

p0=0.01 ca_cfar: scalar=100.7500 ARL0=544

p0=0.01 os_cfar: scalar=166.6660 ARL0=417  UNCALIBRATABLE (bound hit)

p0=0.01 cfar_2of3: scalar=50.8750 ARL0=447

...

p0=0.1 production: scalar=0.2000 ARL0=823  UNCALIBRATABLE (bound hit)

p0=0.1 ca_cfar: scalar=5.2861 ARL0=522

p0=0.1 os_cfar: scalar=3.5327 ARL0=530

p0=0.1 cfar_2of3: scalar=3.5327 ARL0=553

p0=0.1 cusum: scalar=0.1767 ARL0=455

## Shift race and verdict

Step and ramp shifts at doc 300, 200 trials per detector per scenario per noise floor. Delay = docs from onset to the first alarm at or after onset; pre-onset alarms are false alarms already paid for by the matched ARL0 (CUSUM resets on them); miss = no alarm within the 40-doc horizon. The realized wave-1b series then runs through every calibrated detector - the campaign says it is clean, so the silent detectors are the honest ones. A detector that could not calibrate (bound hit) is reported as such and cannot be vindicated.

In [6]:
def shifted_path(p0, kind):
    p = np.full(STREAM_LEN, p0)
    top = min(1.0, p0 + SHIFT_DELTA)
    if kind == "step":
        p[ONSET:] = top
    else:
        for j in range(RAMP_LEN):
            p[ONSET + j] = min(1.0, p0 + SHIFT_DELTA * (j + 1) / RAMP_LEN)
        p[ONSET + RAMP_LEN:] = top
    return p


race = {}
with Progress() as progress:
    task = progress.add_task("racing", total=len(P0_GRID) * 2 * len(DETECTORS))
    for p0 in P0_GRID:
        for kind in ("step", "ramp"):
            trials = [simulate_stream(shifted_path(p0, kind), rng) for _ in range(N_TRIALS)]
            for name, (fn, _, _) in DETECTORS.items():
                scalar = calibrated[(p0, name)]["scalar"]
                delays, misses = [], 0
                for x in trials:
                    a = fn(x, scalar)
                    post = a[(a >= ONSET) & (a <= ONSET + HORIZON)]
                    if len(post):
                        delays.append(int(post[0]) - ONSET)
                    else:
                        misses += 1
                race[(p0, kind, name)] = {
                    "miss_rate": misses / N_TRIALS,
                    "median_delay": float(np.median(delays)) if delays else None,
                }
                progress.advance(task)

clean_alarms = {
    name: ([int(i) for i in fn(real_series, calibrated[(0.05, name)]["scalar"])] or None)
    for name, (fn, _, _) in DETECTORS.items()
}

for p0 in P0_GRID:
    rprint(f"\n[bold cyan]p0 = {p0}[/bold cyan]  [dim](all detectors at ARL0 ~{ARL0_TARGET})[/dim]")
    for kind in ("step", "ramp"):
        rprint(f"  [bold]{kind}[/bold]")
        for name in DETECTORS:
            r = race[(p0, kind, name)]
            tag = "" if calibrated[(p0, name)]["in_band"] else " [red](uncalibratable)[/red]"
            rprint(f"    {name:>10}: miss [yellow]{r['miss_rate']:.2f}[/yellow]  median delay [yellow]{r['median_delay']}[/yellow]{tag}")
rprint(f"\nrealized clean series alarms (should be None): [yellow]{clean_alarms}[/yellow]")

cfar_variants = ["ca_cfar", "os_cfar", "cfar_2of3"]


def _ratio(a, b):
    if a is None:
        return float("inf")
    return a / max(b, 1e-9) if b is not None else 0.0


verdicts = {}
for p0 in P0_GRID:
    cs, cr = race[(p0, "step", "cusum")], race[(p0, "ramp", "cusum")]
    all_refuted, best_ok = True, False
    for v in cfar_variants:
        vs, vr = race[(p0, "step", v)], race[(p0, "ramp", v)]
        calibratable = calibrated[(p0, v)]["in_band"]
        delay_bad = _ratio(vs["median_delay"], cs["median_delay"]) >= 2
        miss_bad = vr["miss_rate"] >= 2 * max(cr["miss_rate"], 0.025)
        if calibratable and not (delay_bad or miss_bad):
            all_refuted = False
        within = (
            calibratable
            and _ratio(vs["median_delay"], cs["median_delay"]) <= 1.25
            and vr["miss_rate"] <= cr["miss_rate"] + 0.25 * max(cr["miss_rate"], 0.04)
        )
        best_ok = best_ok or within
    verdicts[p0] = "CFAR_REFUTED" if all_refuted else ("CFAR_VINDICATED" if best_ok else "MIXED")

overall = (
    "CFAR_REFUTED" if all(v == "CFAR_REFUTED" for v in verdicts.values())
    else "CFAR_VINDICATED" if all(v == "CFAR_VINDICATED" for v in verdicts.values())
    else "MIXED"
)
rprint(f"\n[bold]per-noise verdicts:[/bold] {verdicts}")
rprint(f"[bold]overall:[/bold] [{'red' if overall == 'CFAR_REFUTED' else 'green' if overall == 'CFAR_VINDICATED' else 'yellow'}]{overall}[/]")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"drift-cfar-h50-{stamp}.json"
out.write_text(json.dumps({
    "calibrated": {f"{k[0]}|{k[1]}": v for k, v in calibrated.items()},
    "race": {f"{k[0]}|{k[1]}|{k[2]}": v for k, v in race.items()},
    "clean_series_alarms": clean_alarms,
    "verdicts": {str(k): v for k, v in verdicts.items()}, "overall": overall,
}, indent=2))
rprint("saved", str(out))

p0 = 0.01  (all detectors at ARL0 ~500)

step

production: miss 0.00  median delay 2.0 (uncalibratable)

ca_cfar: miss 0.77  median delay 0.0

os_cfar: miss 0.09  median delay 0.0 (uncalibratable)

...

cusum: miss 0.00  median delay 7.0

realized clean series alarms (should be None): {'production': None, 'ca_cfar': [75], 'os_cfar': [75], 'cfar_2of3': 
None, 'cusum': [75]}

per-noise verdicts: {0.01: 'CFAR_REFUTED', 0.05: 'CFAR_REFUTED', 0.1: 'CFAR_REFUTED'}

overall: CFAR_REFUTED

saved ../reports/drift-cfar-h50-20260706-183747.json